In [1]:
import numpy as np
import glob
import os

# --- Get one of your input images ---
VAL_DIR = "data/UDC-SIT_subset/val"
input_paths = sorted(glob.glob(os.path.join(VAL_DIR, "input", "*.npy")))

if not input_paths:
    print(f"Error: No .npy files found in {os.path.join(VAL_DIR, 'input')}")
else:
    file_path = input_paths[0]
    
    try:
        data = np.load(file_path)
        
        print(f"--- Metadata for: {os.path.basename(file_path)} ---")
        print(f"Data Type (dtype): {data.dtype}")
        print(f"Data Shape: {data.shape}")
        
        # Check the value range
        print(f"Min value: {data.min()}")
        print(f"Max value: {data.max()}")
        print(f"Mean value: {data.mean()}")
        
    except Exception as e:
        print(f"An error occurred: {e}")

--- Metadata for: 1004.npy ---
Data Type (dtype): float32
Data Shape: (1792, 1280, 4)
Min value: 0.0
Max value: 1023.0
Mean value: 129.1371307373047


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from models.mambair_teacher import FrequencyAwareTeacher

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# --- LOAD THE NEW 10-BIT MODEL ---
MODEL_WEIGHTS_PATH = "teacher_10bit_normalized.pth" 
VAL_DIR = "data/UDC-SIT_subset/val"
PATCH_SIZE = 256
# ---------------------

def load_and_process_patch(npy_path, device, patch_size=256):
    """Loads a .npy file, normalizes it, and extracts a center patch."""
    img = np.load(npy_path) # Data is [0, 1023]
    
    # Get a center crop
    H, W, _ = img.shape
    ps = patch_size
    ps_h = min(ps, H)
    ps_w = min(ps, W)
    rr = (H - ps_h) // 2
    rc = (W - ps_w) // 2
    patch = img[rr : rr + ps_h, rc : rc + ps_w, :]

    # Convert patch to tensor
    img_tensor = torch.from_numpy(patch.copy()).permute(2, 0, 1).float()
    
    # --- NORMALIZE THE INPUT ---
    img_tensor /= 1023.0 # Use the correct 10-bit max value
    
    # Add batch dimension
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    # Return tensor and the NORMALIZED patch for display
    return img_tensor, patch / 1023.0 

def post_process_output(output_tensor):
    """Converts model output tensor (already in [0, 1] range) to numpy."""
    img = output_tensor.squeeze(0).detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    # The new model outputs [0, 1], so we just clip
    img = np.clip(img, 0, 1)
    return img

# 1. Load the model
print(f"Loading model from {MODEL_WEIGHTS_PATH}...")
model = FrequencyAwareTeacher(in_channels=4, out_channels=3).to(DEVICE)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH))
model.eval()
print("Model loaded.")

# 2. Get one validation image
input_paths = sorted(glob.glob(os.path.join(VAL_DIR, "input", "*.npy")))
if not input_paths:
    print(f"Error: No .npy files found in {os.path.join(VAL_DIR, 'input')}")
else:
    input_path = input_paths[0]
    gt_path = input_path.replace("/input/", "/GT/")
    
    # 3. Load and preprocess patches
    udc_tensor, udc_patch_np = load_and_process_patch(input_path, DEVICE, PATCH_SIZE)
    _, gt_patch_np = load_and_process_patch(gt_path, DEVICE, PATCH_SIZE)
    
    print(f"Processing center patch of: {os.path.basename(input_path)}")
    
    # 4. Run inference
    with torch.no_grad():
        output_tensor = model(udc_tensor)
        
    # 5. Post-process the output
    restored_img__np = post_process_output(output_tensor)

    # 6. Visualize the results
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    axes[0].imshow(udc_patch_np[:, :, :3])
    axes[0].set_title("Input Patch (UDC)")
    axes[0].axis("off")
    
    axes[1].imshow(restored_img_np)
    axes[1].set_title("Model Output (Restored)")
    axes[1].axis("off")
    
    axes[2].imshow(gt_patch_np[:, :, :3])
    axes[2].set_title("Ground Truth Patch")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()